# KuantLab — Ücretsiz LLM Kuantizasyonu (GGUF)

Bu defter KuantLab sitesini Google Colab üzerinde çalıştırır ve herkese açık bir link verir.

**Kullanım:** Hücreleri sırayla çalıştırın (`Çalışma zamanı > Tümünü çalıştır`). GPU gerekmez; ücretsiz CPU çalışma zamanı yeterlidir.

- Ücretsiz Colab'da ~12–16 GB RAM ve ~80–100 GB disk vardır. Bu alan **~14B parametreye kadar** modeller için yeterlidir.
- Colab oturumu kapandığında site de kapanır ve link değişir (ücretsiz oturumlar en fazla birkaç saat açık kalır).
- Erişim izni gerektiren modeller (ör. `meta-llama/...`) için soldaki **anahtar (Secrets)** menüsüne `HF_TOKEN` ekleyin.

In [ ]:
#@title 1) Projeyi al
#@markdown Projeyi GitHub'a yüklediyseniz adresini yazın. Boş bırakırsanız `kuantlab.zip` dosyasını yüklemeniz istenir.
REPO_URL = ""  #@param {type:"string"}
import os
os.chdir("/content")
if REPO_URL:
    !rm -rf /content/kuantlab && git clone -q $REPO_URL /content/kuantlab
else:
    from google.colab import files
    print("kuantlab.zip dosyasını seçin...")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    !rm -rf /content/kuantlab && mkdir -p /content/kuantlab && unzip -q -o "/content/$zip_name" -d /content/kuantlab
os.chdir("/content/kuantlab")
!ls

In [ ]:
#@title 2) Kurulum (llama.cpp + Python paketleri, ~1-2 dk)
!pip install -q -r requirements.txt
!python -m app.setup_llama

In [ ]:
#@title 3) (İsteğe bağlı) Hugging Face token
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN yüklendi: erişim izni gerektiren modeller indirilebilir.")
except Exception:
    print("HF_TOKEN bulunamadı: herkese açık modeller kullanılabilir (token sitede de girilebilir).")

In [ ]:
#@title 4) Siteyi başlat ve herkese açık link al
import os, re, subprocess, sys, time

os.environ.setdefault("KUANT_MAX_MODEL_GB", "32")
for p in ("server", "tunnel"):
    if p in globals():
        globals()[p].terminate()

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/kuantlab", stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)

if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared

tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
    stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT,
)

url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break

print("Site adresi:", url or "bulunamadı, /content/tunnel.log dosyasına bakın")
print("Bu linki paylaşabilirsiniz. Colab sekmesi açık kaldığı sürece site çalışır.")

In [ ]:
#@title 5) Sunucu günlüğü
!tail -n 40 /content/server.log